<a href="https://colab.research.google.com/github/sharvani1357/RAG/blob/main/chunking_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
#Step 0: Create Employee Policy PDF
!pip install reportlab
from reportlab.pdfgen import canvas

pdf_path = "employee_policy.pdf"

c = canvas.Canvas(pdf_path)

content = """
Leave Policy

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.

Work From Home Policy

Employees may work from home twice per week.
Manager approval is required for additional remote work.

Travel Policy

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.

Medical Insurance Policy

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
"""

y = 800

for line in content.split("\n"):
    c.drawString(50, y, line)
    y -= 20

c.save()

print("PDF Created Successfully!")
print("Saved as:", pdf_path)
#Verify PDF
import os

print("Exists:", os.path.exists("employee_policy.pdf"))
print("Size:", os.path.getsize("employee_policy.pdf"), "bytes")

PDF Created Successfully!
Saved as: employee_policy.pdf
Exists: True
Size: 1798 bytes


In [15]:
!pip install PyPDF2
from PyPDF2 import PdfReader

reader = PdfReader("employee_policy.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print("===== ORIGINAL DOCUMENT =====\n")
print(text)

print("\n===== DOCUMENT STATISTICS =====")
print("Total Characters:", len(text))
print("Total Words:", len(text.split()))

===== ORIGINAL DOCUMENT =====

Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.


===== DOCUMENT STATISTICS =====
Total Characters: 530
Total Words: 76


In [16]:
#Task 2: Fixed Size Chunking
from textwrap import wrap

chunk_sizes = [100, 200, 300]

for size in chunk_sizes:

    print(f"\n{'='*60}")
    print(f"FIXED SIZE CHUNKING (Chunk Size={size})")
    print(f"{'='*60}")

    chunks = wrap(text, size)

    for i, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {i}")
        print("-"*40)
        print(chunk)
        print("Length:", len(chunk))
#Analysis
print("""
ANALYSIS

1. Sentences may be split in the middle.
2. Context can be lost across chunk boundaries.
3. Easy to implement.
4. Poor retrieval quality for long documents.
""")


FIXED SIZE CHUNKING (Chunk Size=100)

Chunk 1
----------------------------------------
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick leaves annually.
Length: 100

Chunk 2
----------------------------------------
Unused casual leaves cannot be carried forward. Work From Home Policy Employees may work from home
Length: 98

Chunk 3
----------------------------------------
twice per week. Manager approval is required for additional remote work. Travel Policy Travel
Length: 93

Chunk 4
----------------------------------------
expenses are reimbursed within 30 days. Original receipts must be submitted for reimbursement.
Length: 94

Chunk 5
----------------------------------------
Medical Insurance Policy All employees are covered under company medical insurance. Dependent
Length: 93

Chunk 6
----------------------------------------
coverage is available for spouse and children.
Length: 46

FIXED SIZE CHUNKING (Chunk Size=200)

Chunk 1
------------------

In [17]:
#Task 3: Recursive Chunking
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

configs = [
    (200,20),
    (200,50),
    (300,50)
]

for chunk_size, overlap in configs:

    print(f"\n{'='*60}")
    print(f"Chunk Size={chunk_size}, Overlap={overlap}")
    print(f"{'='*60}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(text)

    for i, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {i}")
        print("-"*40)
        print(chunk)
#Analysis
print("""
RECURSIVE CHUNKING ANALYSIS

Overlap preserves context between chunks.

Example:
Chunk 1 ends with:
'Employees may work from home'

Chunk 2 starts with:
'work from home twice per week'

This helps retrievers avoid losing important information.
""")


Chunk Size=200, Overlap=20

Chunk 1
----------------------------------------
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy

Chunk 2
----------------------------------------
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.

Chunk 3
----------------------------------------
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.

Chunk Size=200, Overlap=50

Chunk 1
----------------------------------------
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy

Chunk 2
----------------------------------

In [18]:
#Task 4: Sentence-Based Chunking
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt_tab')

sentences = sent_tokenize(text)

print("TOTAL SENTENCES:", len(sentences))

for i, sentence in enumerate(sentences, start=1):
    print(f"\nSentence {i}")
    print("-"*40)
    print(sentence)
#Analysis
print("""
SENTENCE CHUNKING ANALYSIS

1. Sentences remain intact.
2. Meaning is preserved.
3. Better than fixed-size chunking.
4. Context spanning multiple sentences may still be lost.
""")

TOTAL SENTENCES: 9

Sentence 1
----------------------------------------
Leave Policy
Employees receive 12 casual leaves annually.

Sentence 2
----------------------------------------
Employees receive 15 sick leaves annually.

Sentence 3
----------------------------------------
Unused casual leaves cannot be carried forward.

Sentence 4
----------------------------------------
Work From Home Policy
Employees may work from home twice per week.

Sentence 5
----------------------------------------
Manager approval is required for additional remote work.

Sentence 6
----------------------------------------
Travel Policy
Travel expenses are reimbursed within 30 days.

Sentence 7
----------------------------------------
Original receipts must be submitted for reimbursement.

Sentence 8
----------------------------------------
Medical Insurance Policy
All employees are covered under company medical insurance.

Sentence 9
----------------------------------------
Dependent coverage is available

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [19]:
#Task 5: Semantic Chunking

#Since policies are already section-wise.

sections = text.split("Policy")

semantic_chunks = []

for section in sections:
    section = section.strip()

    if section:
        semantic_chunks.append(section)

for i, chunk in enumerate(semantic_chunks, start=1):
    print(f"\nSemantic Chunk {i}")
    print("-"*50)
    print(chunk)
#Better Manual Semantic Chunking
semantic_chunks = {

"Leave Policy":
"""
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
""",

"Work From Home Policy":
"""
Employees may work from home twice per week.
Manager approval is required for additional remote work.
""",

"Travel Policy":
"""
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
""",

"Medical Insurance Policy":
"""
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
"""
}

for title, content in semantic_chunks.items():
    print(f"\n{title}")
    print("-"*50)
    print(content)
#Analysis
print("""
SEMANTIC CHUNKING ANALYSIS

1. Groups related information together.
2. Highest contextual coherence.
3. Better retrieval quality.
4. Preferred for enterprise RAG systems.
""")


Semantic Chunk 1
--------------------------------------------------
Leave

Semantic Chunk 2
--------------------------------------------------
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home

Semantic Chunk 3
--------------------------------------------------
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel

Semantic Chunk 4
--------------------------------------------------
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance

Semantic Chunk 5
--------------------------------------------------
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.

Leave Policy
--------------------------------------------------

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.

In [20]:
import pandas as pd

comparison = pd.DataFrame({

"Chunking Method":[
    "Fixed Size",
    "Recursive",
    "Sentence",
    "Semantic"
],

"Context Preservation":[
    "Low",
    "High",
    "Medium",
    "Very High"
],

"Retrieval Quality":[
    "Low",
    "High",
    "Medium",
    "Very High"
],

"Implementation Complexity":[
    "Easy",
    "Moderate",
    "Easy",
    "High"
]

})

print(comparison)

  Chunking Method Context Preservation Retrieval Quality  \
0      Fixed Size                  Low               Low   
1       Recursive                 High              High   
2        Sentence               Medium            Medium   
3        Semantic            Very High         Very High   

  Implementation Complexity  
0                      Easy  
1                  Moderate  
2                      Easy  
3                      High  


In [21]:
#Task 7: Retrieval Simulation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

queries = [
    "How many casual leaves are provided?",
    "Can employees work from home?",
    "What is the travel reimbursement process?",
    "Who is covered under medical insurance?"
]

# Example using semantic chunks

chunks = list(semantic_chunks.values())

vectorizer = TfidfVectorizer()

chunk_vectors = vectorizer.fit_transform(chunks)

for query in queries:

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        chunk_vectors
    )

    best_match = similarity.argmax()

    print(f"\nQuery: {query}")
    print("\nRetrieved Chunk:")
    print(chunks[best_match])

    print("\nChunking Method: Semantic")
    print("="*60)


Query: How many casual leaves are provided?

Retrieved Chunk:

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.


Chunking Method: Semantic

Query: Can employees work from home?

Retrieved Chunk:

Employees may work from home twice per week.
Manager approval is required for additional remote work.


Chunking Method: Semantic

Query: What is the travel reimbursement process?

Retrieved Chunk:

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.


Chunking Method: Semantic

Query: Who is covered under medical insurance?

Retrieved Chunk:

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.


Chunking Method: Semantic


In [22]:
#Task 8: Recommendation Report
print("""
==================================================
RECOMMENDATION REPORT
==================================================

1. HR POLICY ASSISTANT
Recommended: Semantic Chunking

Reason:
Policies are naturally divided into sections.
Retrieval accuracy is very high.

--------------------------------------------------

2. LEGAL DOCUMENT ASSISTANT
Recommended: Recursive Chunking

Reason:
Legal clauses are interconnected.
Overlap preserves clause references.

--------------------------------------------------

3. MEDICAL DOCUMENT ASSISTANT
Recommended: Semantic Chunking

Reason:
Medical topics should remain grouped.
Maintains clinical context.

--------------------------------------------------

4. RESEARCH PAPER ASSISTANT
Recommended: Recursive Chunking

Reason:
Research concepts span multiple paragraphs.
Overlap improves contextual continuity.

--------------------------------------------------

FINAL CONCLUSION

Semantic Chunking:
Best Retrieval Quality

Recursive Chunking:
Best General-Purpose RAG Strategy

Enterprise Systems:
Semantic + Recursive Hybrid Approach
is considered the most effective.
==================================================
""")


RECOMMENDATION REPORT

1. HR POLICY ASSISTANT
Recommended: Semantic Chunking

Reason:
Policies are naturally divided into sections.
Retrieval accuracy is very high.

--------------------------------------------------

2. LEGAL DOCUMENT ASSISTANT
Recommended: Recursive Chunking

Reason:
Legal clauses are interconnected.
Overlap preserves clause references.

--------------------------------------------------

3. MEDICAL DOCUMENT ASSISTANT
Recommended: Semantic Chunking

Reason:
Medical topics should remain grouped.
Maintains clinical context.

--------------------------------------------------

4. RESEARCH PAPER ASSISTANT
Recommended: Recursive Chunking

Reason:
Research concepts span multiple paragraphs.
Overlap improves contextual continuity.

--------------------------------------------------

FINAL CONCLUSION

Semantic Chunking:
Best Retrieval Quality

Recursive Chunking:
Best General-Purpose RAG Strategy

Enterprise Systems:
Semantic + Recursive Hybrid Approach
is considered the mo

# EMBEDDINGS


In [23]:
document = """
Leave Policy

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.

Work From Home Policy

Employees may work from home twice per week.
Manager approval is required for additional remote work.

Travel Policy

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.

Medical Insurance Policy

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
"""

In [24]:
print("-" * 50)
print("ORIGINAL DOCUMENT")
print("-" * 50)
print(document)

--------------------------------------------------
ORIGINAL DOCUMENT
--------------------------------------------------

Leave Policy
 
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
 
Work From Home Policy
 
Employees may work from home twice per week.
Manager approval is required for additional remote work.
 
Travel Policy
 
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
 
Medical Insurance Policy
 
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.



In [25]:


#Recursive chunks
splitter=RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)
chunks=splitter.split_text(document)

print("\n")
print("="*80)
print("Generated chunks")
print("="*80)
for i,chunk in enumerate(chunks,start=1):
  print(f"chunk {i}:{chunk}")
  print('-'*50)



Generated chunks
chunk 1:Leave Policy
 
Employees receive 12 casual leaves annually.
--------------------------------------------------
chunk 2:Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
--------------------------------------------------
chunk 3:Work From Home Policy
 
Employees may work from home twice per week.
--------------------------------------------------
chunk 4:Manager approval is required for additional remote work.
 
Travel Policy
--------------------------------------------------
chunk 5:Travel Policy
 
Travel expenses are reimbursed within 30 days.
--------------------------------------------------
chunk 6:Original receipts must be submitted for reimbursement.
 
Medical Insurance Policy
--------------------------------------------------
chunk 7:All employees are covered under company medical insurance.
--------------------------------------------------
chunk 8:Dependent coverage is available for spouse and children.
------

## Load Embedding Model
### all-MiniLM-L6-v2
#### output vector dimension=384

In [27]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model=SentenceTransformer("all-MiniLM-L6-v2")
print("\n Embedding model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


 Embedding model loaded successfully


generating embeddings

In [28]:
chunk_embeddings=model.encode(chunks)
print("\n Embeddings Generated")


 Embeddings Generated


Display Embedding Information

In [30]:
print("\n")
print("="*80)
print("Embedding Details")
print("="*80)
print("no.of chunks:",len(chunks))
print("embedding shape:",chunk_embeddings.shape)




Embedding Details
no.of chunks: 8
embedding shape: (8, 384)


In [32]:
print("\n")
print("="*80)
print("1st chunk")
print("="*80)
print(chunks[0])



1st chunk
Leave Policy
 
Employees receive 12 casual leaves annually.


In [33]:
print("\n")
print("="*80)
print("First 20 embeddings value")
print("="*80)
print(chunk_embeddings[0][:20])



First 20 embeddings value
[ 0.06098085  0.02667808  0.01290444  0.02433847  0.09787532  0.09084513
  0.00728967 -0.03919937 -0.05303925  0.02543473  0.09522843  0.04315798
 -0.04351875 -0.03531917 -0.00686681  0.04278303 -0.06461599 -0.02985377
 -0.00127596 -0.06867141]


In [34]:
print("\n")
print("-" * 50)
print("CHUNK TO VECTOR MAPPING")
print("-" * 50)

for i, chunk in enumerate(chunks):
    print(f'Chunk {i}')
    print(chunk)
    print("\nVector Shape:")
    print(chunk_embeddings[i].shape)

print("-" * 50)



--------------------------------------------------
CHUNK TO VECTOR MAPPING
--------------------------------------------------
Chunk 0
Leave Policy
 
Employees receive 12 casual leaves annually.

Vector Shape:
(384,)
Chunk 1
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.

Vector Shape:
(384,)
Chunk 2
Work From Home Policy
 
Employees may work from home twice per week.

Vector Shape:
(384,)
Chunk 3
Manager approval is required for additional remote work.
 
Travel Policy

Vector Shape:
(384,)
Chunk 4
Travel Policy
 
Travel expenses are reimbursed within 30 days.

Vector Shape:
(384,)
Chunk 5
Original receipts must be submitted for reimbursement.
 
Medical Insurance Policy

Vector Shape:
(384,)
Chunk 6
All employees are covered under company medical insurance.

Vector Shape:
(384,)
Chunk 7
Dependent coverage is available for spouse and children.

Vector Shape:
(384,)
--------------------------------------------------
